In [ ]:
#Highest
# SCD2 Person + Contact Roles + Auth (Parquet)
# Uses: dfPerson, dfBusinessEntityContact, dfContactType, dfPassword
# Writes:

# …/dim_person_scd2/ (SCD Type‑2 with Type‑1 field mixed in)
# …/person_contactrole/ (deduped NK merge)
# …/person_auth_metadata/ (policy flags)
# …/_run_summaries/person/ (run metrics)


# Highlights: Manual SCD2 over Parquet (no Delta), deterministic surrogate keys (person_sk from BK+version number), Type‑1/Type‑2 mix, optional late-arrival handling (off by default), and run summaries.

In [98]:
# =========================
# DEMO PROLOGUE (same as Notebook 3)
# =========================

from pyspark.sql import functions as F, Window

demo_mode = True
enable_incremental = False
allow_late_arrival = False         # If True, you'd implement backdating behavior; off for demo.
enable_pit_snapshot = False        # Optional point-in-time output; off for demo.

silver_base_path = "abfss://fabmigation1@synpasetofabric.dfs.core.windows.net/silver/adventureworks"

def _can_write(path: str) -> bool:
    try:
        test_df = spark.createDataFrame([(1,)], ["x"])
        test_tmp = path.rstrip("/") + "/_write_probe__"
        test_df.write.mode("overwrite").parquet(test_tmp)
        try:
            spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration()) \
                .delete(spark._jvm.org.apache.hadoop.fs.Path(test_tmp), True)
        except Exception:
            pass
        return True
    except Exception:
        return False

if _can_write(silver_base_path):
    out_base = silver_base_path
else:
    out_base = silver_base_path

print(f"[INFO] Using output base: {out_base}")

In [99]:

def parquet_exists(path: str) -> bool:
    try:
        spark.read.parquet(path).limit(1).count()
        return True
    except Exception:
        return False

def read_parquet_if_exists(path: str):
    return spark.read.parquet(path) if parquet_exists(path) else None


In [100]:

# =========================
# OUTPUT PATHS
# =========================
out_dim_person     = f"{out_base}/dim_person_scd2"
out_contactrole    = f"{out_base}/person_contactrole"
out_authmeta       = f"{out_base}/person_auth_metadata"
out_summary_person = f"{out_base}/_run_summaries/person"
# Optional PIT output if enabled
out_person_pit     = f"{out_base}/dim_person_pit"


In [101]:

# =========================
# INPUTS
# =========================
# dfPerson: Person.Person
# dfBusinessEntityContact: Person.BusinessEntityContact
# dfContactType: Person.ContactType
# dfPassword: Person.Password
base = "abfss://fabmigation1@synpasetofabric.dfs.core.windows.net/bronze/adventureworks/Person"

tables = ["Person","BusinessEntityContact","ContactType","Password"]

dfs = {t: spark.read.parquet(f"{base}.{t}.parquet/") for t in tables}

dfPerson = dfs["Person"]; dfBusinessEntityContact = dfs["BusinessEntityContact"]; dfContactType = dfs["ContactType"]
dfPassword = dfs["Password"]
display(dfPassword)

In [102]:
# =========================
# 1) SCD2 (with mixed Type-1 field) for Person
# =========================

# Define which attributes are Type-2 vs Type-1
t2_attrs = ["PersonType","NameStyle","Title","FirstName","MiddleName","LastName","Suffix","AdditionalContactInfo"]
t1_attrs = ["EmailPromotion"]  # overwrite-in-place on current version

# Prepare incoming, normalize names, compute hashdiff for T2 attrs only
incoming_full = (
    dfPerson.select(
        F.col("BusinessEntityID").alias("business_entity_id"),
        "PersonType","NameStyle","Title","FirstName","MiddleName","LastName","Suffix",
        "EmailPromotion","AdditionalContactInfo","rowguid","ModifiedDate"
    )
    .withColumn("FirstName", F.initcap(F.col("FirstName")))
    .withColumn("MiddleName", F.initcap(F.col("MiddleName")))
    .withColumn("LastName", F.initcap(F.col("LastName")))
    .withColumn("t2_hashdiff", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in t2_attrs]), 256))
)

if enable_incremental and watermark_from:
    incoming_full = incoming_full.filter(F.col("ModifiedDate") >= F.to_timestamp(F.lit(watermark_from)))

existing = read_parquet_if_exists(out_dim_person)

# Helper to compute deterministic SK from BK+version_no
def compute_person_sk(df):
    return df.withColumn("person_sk", F.sha2(F.concat_ws("||",
                                                         F.col("business_entity_id").cast("string"),
                                                         F.lpad(F.col("version_no").cast("string"), 4, "0")), 256))

if existing is None:
    # Seed: version_no = 1 for all
    seed = (incoming_full
            .withColumn("version_no", F.lit(1))
            .withColumn("effective_start_date", F.current_timestamp())
            .withColumn("effective_end_date", F.to_timestamp(F.lit("9999-12-31 23:59:59")))
            .withColumn("is_current", F.lit(True))
           )
    seed = compute_person_sk(seed)
    dim_seed = seed.select(
        "person_sk","business_entity_id", *t2_attrs, *t1_attrs, "rowguid","ModifiedDate",
        "t2_hashdiff","version_no","effective_start_date","effective_end_date","is_current"
    )
    dim_seed.write.mode("overwrite").parquet(out_dim_person)
    display(dim_seed.limit(50))

    scd_summary = spark.createDataFrame([(
        int(dim_seed.count()),  # scd2_new
        0,                      # scd2_changed
        0,                      # scd2_expired
        0,                      # scd2_t1_updates
        0                       # scd2_unchanged
    )], ["scd2_new","scd2_changed","scd2_expired","scd2_t1_updates","scd2_unchanged"]) \
    .withColumn("entity", F.lit("dim_person_scd2")) \
    .withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
    .withColumn("run_ts", F.current_timestamp())
    scd_summary.write.mode("append").parquet(out_summary_person)
    display(scd_summary)

else:
    # Split existing into current vs history
    hist = existing.filter(F.col("is_current") == False)
    curr = existing.filter(F.col("is_current") == True) \
                   .select("person_sk","business_entity_id", *t2_attrs, *t1_attrs,
                           "rowguid","ModifiedDate","t2_hashdiff","version_no",
                           "effective_start_date","effective_end_date","is_current")

    # Join incoming with current to detect changes
    staged = (incoming_full.alias("i")
              .join(curr.alias("c"), F.col("i.business_entity_id")==F.col("c.business_entity_id"), "left")
              .select("i.*",
                      F.col("c.person_sk").alias("curr_sk"),
                      F.col("c.t2_hashdiff").alias("curr_hash"),
                      *[F.col(f"c.{col}").alias(f"curr_{col}") for col in t1_attrs + t2_attrs],
                      F.col("c.version_no").alias("curr_version_no"))
             )

    # Change detection
    t2_changed   = staged.filter( (F.col("curr_hash").isNotNull()) & (F.col("t2_hashdiff") != F.col("curr_hash")) )
    t1_only      = staged.filter( (F.col("curr_hash").isNotNull()) & (F.col("t2_hashdiff") == F.col("curr_hash")) &
                                  F.expr(" OR ".join([f"i.{c} <> curr_{c}" for c in t1_attrs])) )
    unchanged    = staged.filter( (F.col("curr_hash").isNotNull()) & (F.col("t2_hashdiff") == F.col("curr_hash")) &
                                  F.expr(" AND ".join([f"(i.{c} = curr_{c} OR (i.{c} IS NULL AND curr_{c} IS NULL))" for c in t1_attrs])) )
    newkeys      = staged.filter(F.col("curr_hash").isNull())

    # Expire current rows for T2 changes
    changed_keys = t2_changed.select("business_entity_id").dropDuplicates()
    expired_curr = (curr.alias("c")
                    .join(changed_keys.alias("k"), "business_entity_id", "inner")
                    .withColumn("effective_end_date", F.current_timestamp())
                    .withColumn("is_current", F.lit(False)))

    # Insert new versions for T2 changes (version_no = curr_version_no + 1)
    t2_changed_nextver = (t2_changed
                          .withColumn("version_no", F.coalesce(F.col("curr_version_no"), F.lit(0)) + F.lit(1))
                          .withColumn("effective_start_date", F.current_timestamp())
                          .withColumn("effective_end_date", F.to_timestamp(F.lit("9999-12-31 23:59:59")))
                          .withColumn("is_current", F.lit(True)))
    t2_changed_nextver = compute_person_sk(t2_changed_nextver)
    t2_changed_insert = t2_changed_nextver.select(
        "person_sk","business_entity_id", *t2_attrs, *t1_attrs, "rowguid","ModifiedDate",
        F.col("t2_hashdiff").alias("t2_hashdiff"), "version_no",
        "effective_start_date","effective_end_date","is_current"
    )

    # Insert brand-new keys (version_no = 1)
    newkeys_insert = (newkeys
                      .withColumn("version_no", F.lit(1))
                      .withColumn("effective_start_date", F.current_timestamp())
                      .withColumn("effective_end_date", F.to_timestamp(F.lit("9999-12-31 23:59:59")))
                      .withColumn("is_current", F.lit(True)))
    newkeys_insert = compute_person_sk(newkeys_insert)
    newkeys_insert = newkeys_insert.select(
        "person_sk","business_entity_id", *t2_attrs, *t1_attrs, "rowguid","ModifiedDate",
        F.col("t2_hashdiff").alias("t2_hashdiff"), "version_no",
        "effective_start_date","effective_end_date","is_current"
    )

    # Type-1 updates: overwrite EmailPromotion on current versions, no new row
    if len(t1_attrs) > 0:
        t1_updates = curr.alias("c").join(
            t1_only.select("business_entity_id", *[F.col(f"i.{c}").alias(c) for c in t1_attrs]).alias("u"),
            "business_entity_id", "inner"
        )
        for c in t1_attrs:
            t1_updates = t1_updates.withColumn(c, F.col(f"u.{c}"))
        t1_updates = t1_updates.drop(*[f"u.{c}" for c in t1_attrs])
    else:
        t1_updates = spark.createDataFrame([], curr.schema)

    # Current rows that are not T2-changed and not T1-updated remain as-is
    unaffected_curr = (curr.alias("c")
                       .join(changed_keys.alias("k"), "business_entity_id", "left_anti")
                       .join(t1_only.select("business_entity_id").dropDuplicates().alias("t1"),
                             "business_entity_id", "left_anti"))

    # Final SCD2 set
    final_dim = (hist
                 .unionByName(expired_curr, allowMissingColumns=True)
                 .unionByName(unaffected_curr, allowMissingColumns=True)
                 .unionByName(t1_updates, allowMissingColumns=True)
                 .unionByName(t2_changed_insert, allowMissingColumns=True)
                 .unionByName(newkeys_insert, allowMissingColumns=True))

    final_dim.write.mode("overwrite").parquet(out_dim_person)
    display(final_dim.limit(50))

    # Run summary
    scd_summary = spark.createDataFrame([(
        int(newkeys_insert.count()),
        int(t2_changed_insert.count()),
        int(expired_curr.count()),
        int(t1_updates.count()),
        int(unaffected_curr.count())
    )], ["scd2_new","scd2_changed","scd2_expired","scd2_t1_updates","scd2_unchanged"]) \
    .withColumn("entity", F.lit("dim_person_scd2")) \
    .withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
    .withColumn("run_ts", F.current_timestamp())

    scd_summary.write.mode("append").parquet(out_summary_person)
    display(scd_summary)

In [103]:
# =========================
# 2) Contact role bridge (dedup by NK, file-based merge)
# =========================

ct = dfContactType.select("ContactTypeID", F.col("Name").alias("contact_type_name"))

contact_raw = (dfBusinessEntityContact.alias("bec")
               .join(ct.alias("ct"), "ContactTypeID", "left")
               .select(
                   F.col("bec.BusinessEntityID").alias("business_entity_id"),
                   F.col("bec.PersonID").alias("contact_person_id"),
                   F.col("ct.contact_type_name"),
                   F.col("bec.ModifiedDate")
               ))

# Merge with existing by NK (last-wins by ModifiedDate)
existing_contact = read_parquet_if_exists(out_contactrole)

contact_cur = (contact_raw
               .withColumn("nk", F.concat_ws("||",
                                             F.col("business_entity_id").cast("string"),
                                             F.col("contact_person_id").cast("string"),
                                             F.coalesce(F.col("contact_type_name"), F.lit("")))))

if existing_contact is None:
    merged_contact = contact_cur
else:
    merged_contact = existing_contact.unionByName(contact_cur, allowMissingColumns=True)

w_contact = Window.partitionBy("nk").orderBy(F.col("ModifiedDate").desc_nulls_last())
person_contactrole = (merged_contact.withColumn("rn", F.row_number().over(w_contact))
                      .filter("rn = 1")
                      .drop("rn")
                      .withColumn("person_contactrole_sk", F.sha2(F.col("nk"), 256))
                      .drop("nk"))

person_contactrole.write.mode("overwrite").parquet(out_contactrole)

# Run summary for contact roles
summary_contact = spark.createDataFrame([(
    int(contact_raw.count()),
    int(person_contactrole.count()),
    int(person_contactrole.select("business_entity_id").distinct().count())
)], ["raw_links","curated_links","distinct_business_entities"]) \
.withColumn("entity", F.lit("person_contactrole")) \
.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
.withColumn("run_ts", F.current_timestamp())

summary_contact.write.mode("append").parquet(out_summary_person)
display(summary_contact)

In [104]:
display(person_contactrole.limit(50))

In [105]:
# =========================
# 3) Auth metadata (policy flags), file-merge by BK
# =========================

auth_raw = (dfPassword
            .select(
                F.col("BusinessEntityID").alias("business_entity_id"),
                F.length(F.col("PasswordHash")).alias("password_hash_length"),
                F.length(F.col("PasswordSalt")).alias("password_salt_length"),
                "ModifiedDate"
            )
            .withColumn("has_password", (F.col("password_hash_length") > 0) & (F.col("password_salt_length") > 0))
)

# Simple policy check (demo-safe): hash length >= 32 if present
auth = (auth_raw
        .withColumn("policy_violation",
                    F.when( (F.col("has_password") == True) & (F.col("password_hash_length") < 32), F.lit(True))
                     .otherwise(F.lit(False)))
        .withColumn("person_auth_metadata_sk",
                    F.sha2(F.concat_ws("||",
                                       F.col("business_entity_id").cast("string"),
                                       F.coalesce(F.col("password_hash_length").cast("string"), F.lit("0")),
                                       F.coalesce(F.col("password_salt_length").cast("string"), F.lit("0"))
                                      ), 256))
)

existing_auth = read_parquet_if_exists(out_authmeta)
if existing_auth is None:
    merged_auth = auth
else:
    merged_auth = (existing_auth.unionByName(auth, allowMissingColumns=True)
                   .withColumn("nk", F.col("business_entity_id")))

    w_auth = Window.partitionBy("nk").orderBy(F.col("ModifiedDate").desc_nulls_last())
    merged_auth = merged_auth.withColumn("rn", F.row_number().over(w_auth)) \
                             .filter("rn = 1") \
                             .drop("rn","nk")

merged_auth.write.mode("overwrite").parquet(out_authmeta)
display(merged_auth.limit(50))

# Run summary for auth
summary_auth = spark.createDataFrame([(
    int(auth_raw.count()),
    int(merged_auth.count()),
    int(merged_auth.filter(F.col("policy_violation")==True).count())
)], ["raw_rows","curated_rows","policy_violations"]) \
.withColumn("entity", F.lit("person_auth_metadata")) \
.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")) \
.withColumn("run_ts", F.current_timestamp())

summary_auth.write.mode("append").parquet(out_summary_person)
display(summary_auth)
print("[OK] Notebook 4 completed.")
